### An Encoder-Decoder Network for Neural Machine Translation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset
import tokenizers

In [2]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"]
)
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [3]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [4]:
# tokenize
def train_eng_spa(): # a generator function to iterate over all training text
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"]
)
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [5]:
nmt_tokenizer.encode("I like soccer").ids

[43, 401, 4381]

In [6]:
nmt_tokenizer.encode("<s> Me gusta el fútbol").ids

[2, 396, 582, 219, 3356]

In [7]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device: str) -> "NmtPair":
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))


In [ ]:
# Create the data loaders
def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    # the function drops the EoS token from the decoder inputs
    # and drops the SoS token from the decoder targets.
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)


In [9]:
# Build the translation model
class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, pair: NmtPair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0, 2, 1)
    

device = "mps"
torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device)

In [10]:
def evaluate_tm(model, data_loader, metric, device=None):
    if device:
        model = model.to(device)
        metric = metric.to(device)
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end


def train_with_early_stopping(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, early_stopping=None, scheduler=None, device=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)    # forward pass
            loss = criterion(y_pred, y_batch)    # compute loss for this batch
            total_loss = total_loss +  loss.item()
            loss.backward()    # compute gradients of the loss
            optimizer.step()   # adjust the model weights based on the gradients
            optimizer.zero_grad()    # zero the gradients for next batch
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric, device=device).item()
        history["valid_metrics"].append(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
        # Check early stopping condition
        if early_stopping:
            early_stopping.check_early_stop(val_metric)
            if early_stopping.stop_training:
                print(f"Early stopping at epoch {epoch}")
                break
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_metric)
            else:
                scheduler.step()
            print(f"Learning rate: {scheduler.get_last_lr()[0]:.5f}")
    return history

In [11]:
import torchmetrics

n_epochs = 10
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train_with_early_stopping(nmt_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs, device=device)

Epoch 1/10, train loss: 3.1342, train metric: 0.1733, valid metric: 0.2031
Epoch 2/10, train loss: 2.0416, train metric: 0.2195, valid metric: 0.2151
Epoch 3/10, train loss: 1.7222, train metric: 0.2351, valid metric: 0.2176
Epoch 4/10, train loss: 1.5615, train metric: 0.2422, valid metric: 0.2151
Epoch 5/10, train loss: 1.4698, train metric: 0.2469, valid metric: 0.2142
Epoch 6/10, train loss: 1.4188, train metric: 0.2487, valid metric: 0.2135
Epoch 7/10, train loss: 1.3918, train metric: 0.2493, valid metric: 0.2121
Epoch 8/10, train loss: 1.3790, train metric: 0.2493, valid metric: 0.2106
Epoch 9/10, train loss: 1.3771, train metric: 0.2501, valid metric: 0.2117
Epoch 10/10, train loss: 1.3798, train metric: 0.2494, valid metric: 0.2096


In [12]:
torch.save(nmt_model.state_dict(), "my_nmt_model.pt")

In [13]:
def translate(model: nn.Module, src_text: str, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text, "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index] # take tha last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break

    return tgt_text


In [14]:
nmt_model.eval()
translate(nmt_model, "I like soccer.")

' Me gusta el fútbol . </s>'

In [16]:
longer_text = "I like to play soccer with my friends."
translate(nmt_model, longer_text)

' Me gusta jugar fútbol con los adoles c entes . </s>'

### Beam Search

In [17]:
def beam_search(model, src_text, beam_width=3, max_length=20,
                verbose=False, length_penalty=0.6):
    top_translations = [(torch.tensor(0.), "")]
    for index in range(max_length):
        if verbose:
            print(f"Top {beam_width} translations so far:")
            for log_proba, tgt_text in top_translations:
                print(f"    {log_proba.item():.3f} – {tgt_text}")

        candidates = []
        for log_proba, tgt_text in top_translations:
            if tgt_text.endswith(" </s>"):
                candidates.append((log_proba, tgt_text))
                continue  # don't add tokens after EOS token
            batch, _ = nmt_collate_fn([{"source_text": src_text,
                                        "target_text": tgt_text}])
            with torch.no_grad():
                Y_logits = model(batch.to(device))
                Y_log_proba = F.log_softmax(Y_logits, dim=1)
                Y_top_log_probas = torch.topk(Y_log_proba, k=beam_width, dim=1)

            for beam_index in range(beam_width):
                next_token_log_proba = Y_top_log_probas.values[0, beam_index, index]
                next_token_id = Y_top_log_probas.indices[0, beam_index, index]
                next_token = nmt_tokenizer.id_to_token(next_token_id)
                next_tgt_text = tgt_text + " " + next_token
                candidates.append((log_proba + next_token_log_proba, next_tgt_text))

        def length_penalized_score(candidate, alpha=length_penalty):
            log_proba, text = candidate
            length = len(text.split())
            penalty = ((5 + length) ** alpha) / (6 ** alpha)
            return log_proba / penalty

        top_translations = sorted(candidates,
                                  key=length_penalized_score,
                                  reverse=True)[:beam_width]

    return top_translations[-1][1]

In [18]:
beam_search(nmt_model, longer_text, beam_width=3)

' Me gusta jugar al fútbol con los adoles c entes amigos . </s>'

In [19]:
longest_text = "I like to play soccer with my friends at the beach."
beam_search(nmt_model, longer_text, beam_width=3)

' Me gusta jugar al fútbol con los adoles c entes amigos . </s>'

In [21]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

In [22]:
del_vars(["nmt_model"])

### Attention mechanisms

In [ ]:
# Luong attention (dot-product)
def attention(query, key, value):  # note: dq == dk and Lk == Lv
    scores = query @ key.transpose(1, 2)  # [B,Lq,dq] @ [B,dk,Lk] = [B, Lq, Lk]
    weights = torch.softmax(scores, dim=-1)  # [B, Lq, Lk]
    return weights @ value  # [B, Lq, Lk] @ [B, Lv, dv] = [B, Lq, dv]

* `B` = batch size
* `Lq` = max query length in this batch
* `Lk` = max key length in this batch = `Lv` = max value length in this batch
* `dq` = query embedding size = `dk` = key embedding size
* `dv` = value embedding size

In [24]:
class NmtModelWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512,
                 n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        # double size since we concatenate the attention vectors to the decoder outputs
        self.output = nn.Linear(2 * hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)  # same as earlier
        tgt_embeddings = self.embed(pair.tgt_token_ids)  # same
        src_lengths = pair.src_mask.sum(dim=1)  # same
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False)  # same
        encoder_outputs_packed, hidden_states = self.encoder(src_packed)
        decoder_outputs, _ = self.decoder(tgt_embeddings, hidden_states)  # same
        encoder_outputs, _ = pad_packed_sequence(encoder_outputs_packed,
                                                 batch_first=True)
        attn_output = attention(query=decoder_outputs,
                                key=encoder_outputs, value=encoder_outputs)
        combined_output = torch.cat((attn_output, decoder_outputs), dim=-1)
        return self.output(combined_output).permute(0, 2, 1)

In [25]:
torch.manual_seed(42)
nmt_attn_model = NmtModelWithAttention(vocab_size).to(device)

n_epochs = 10
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_attn_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train_with_early_stopping(nmt_attn_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs, device=device)

Epoch 1/10, train loss: 3.0073, train metric: 0.1803, valid metric: 0.2051
Epoch 2/10, train loss: 2.1260, train metric: 0.2160, valid metric: 0.2104
Epoch 3/10, train loss: 1.9269, train metric: 0.2252, valid metric: 0.2146
Epoch 4/10, train loss: 1.8351, train metric: 0.2293, valid metric: 0.2119
Epoch 5/10, train loss: 1.7901, train metric: 0.2317, valid metric: 0.2138
Epoch 6/10, train loss: 1.7598, train metric: 0.2324, valid metric: 0.2138
Epoch 7/10, train loss: 1.7444, train metric: 0.2328, valid metric: 0.2114
Epoch 8/10, train loss: 1.7375, train metric: 0.2328, valid metric: 0.2126
Epoch 9/10, train loss: 1.7261, train metric: 0.2346, valid metric: 0.2126
Epoch 10/10, train loss: 1.7226, train metric: 0.2347, valid metric: 0.2123


In [27]:
translate(nmt_attn_model, longest_text)

' Me gusta jugar al fútbol con mis amigos . </s>'

In [28]:
beam_search(nmt_attn_model, longest_text)

' Me gusta jugar al fútbol con mis compañeros . </s>'